# Cross-Industry Accelerator — Create Semantic Model
### Auto-Generate Power BI Semantic Model from Lakehouse or Warehouse

This notebook creates a **Power BI Semantic Model** (dataset) by:

1. Reading the dynamic table/schema discovery from `00_Industry_Config`
2. Letting you choose the **data source**: Lakehouse (Delta tables) or Warehouse (SQL tables)
3. Building a TMSL (Tabular Model Scripting Language) definition with:
   - All dimension and fact tables as model tables
   - Auto-detected relationships (FK columns matching dim PK patterns)
   - Auto-generated measures for numeric fact columns (SUM, AVG, COUNT)
   - Date hierarchies on detected date columns
4. Creating the semantic model in Fabric via the REST API

> **Prerequisites:**
> 1. Run `02_Load_Lakehouse.ipynb` and/or `03_Load_Warehouse.ipynb` so tables exist
> 2. The Fabric workspace must have Power BI capacity assigned

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════
%run ./00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# READ TABLE SCHEMAS FROM CSVs
# ════════════════════════════════════════════════════════════════════════

import pandas as pd
from pyspark.sql.types import (
    StringType, IntegerType, LongType, FloatType, DoubleType,
    BooleanType, DateType, TimestampType, DecimalType
)

# Spark type → TMSL data type mapping
SPARK_TO_TMSL = {
    StringType:    "string",
    IntegerType:   "int64",
    LongType:      "int64",
    FloatType:     "double",
    DoubleType:    "double",
    BooleanType:   "boolean",
    DateType:      "dateTime",
    TimestampType: "dateTime",
    DecimalType:   "decimal",
}

# Read schemas for all tables (dim + fact + event + stream)
# ── Choose data source: "lakehouse" or "warehouse" ──────────────────
SEMANTIC_MODEL_SOURCE = "lakehouse"   # "lakehouse" → Delta tables  |  "warehouse" → SQL tables

print(f"Semantic Model Source: {SEMANTIC_MODEL_SOURCE.upper()}")
if SEMANTIC_MODEL_SOURCE == "lakehouse":
    print(f"  → Tables will be bound to Lakehouse: {LAKEHOUSE_NAME}")
else:
    print(f"  → Tables will be bound to Warehouse: {WAREHOUSE_NAME}")

ALL_TABLES = DIM_TABLES + FACT_TABLES_BATCH + FACT_TABLES_EVENT + STREAM_TABLES
table_schemas = {}

print(f"Reading schemas for {len(ALL_TABLES)} tables...\n")
for table_name in ALL_TABLES:
    csv_path = f"{CSV_BASE_PATH}/{table_name}.csv"
    try:
        df = (spark.read

              .option("header", True)print(f"\n✅ Schemas loaded for {len(table_schemas)} tables.")

              .option("inferSchema", True)

              .csv(csv_path))        print(f"  ✗ {table_name:<45} ERROR: {e}")

        table_schemas[table_name] = df.schema    except Exception as e:
        print(f"  ✓ {table_name:<45} {len(df.schema.fields)} columns")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# GENERATE TMSL MODEL DEFINITION (ZT-hardened)
# ════════════════════════════════════════════════════════════════════════

import json

def build_tmsl_column(field):
    """Build a TMSL column definition from a Spark StructField."""
    tmsl_type = SPARK_TO_TMSL.get(type(field.dataType), "string")
    col = {
        "name": field.name,
        "dataType": tmsl_type,
        "sourceColumn": field.name,
    }
    # Mark likely ID columns as key
    if field.name.lower().endswith("_id") and field.name == sorted(
        [f.name for f in table_schemas.get(current_table, []) if f.name.lower().endswith("_id")]
    )[0]:
        col["isKey"] = True
    return col

def detect_relationships(table_schemas, dim_tables):
    """Auto-detect star-schema relationships: fact FK → dim PK."""
    relationships = []
    dim_pks = {}
    # Detect PK for each dim table (first _id column)
    for dim in dim_tables:
        if dim in table_schemas:
            id_cols = [f.name for f in table_schemas[dim].fields if f.name.lower().endswith("_id")]
            if id_cols:
                dim_pks[dim] = id_cols[0]

    # Find matching FK columns in fact/event tables
    fact_tables = [t for t in table_schemas if t not in dim_tables]
    rel_id = 0
    for fact in fact_tables:
        fact_cols = [f.name for f in table_schemas[fact].fields]
        for dim, pk in dim_pks.items():
            if pk in fact_cols:
                rel_id += 1
                relationships.append({
                    "name": f"rel_{rel_id}_{fact}_{dim}",
                    "fromTable": fact,
                    "fromColumn": pk,
                    "toTable": dim,
                    "toColumn": pk,
                })
    return relationships

def generate_measures(table_name, schema):
    """Generate SUM/AVG/COUNT measures for numeric columns in fact tables.
    ZT: Excludes PII/PHI-sensitive columns from auto-generated measures."""
    measures = []
    if not table_name.startswith("dim_"):
        numeric_types = (IntegerType, LongType, FloatType, DoubleType, DecimalType)
        for field in schema.fields:
            if isinstance(field.dataType, numeric_types) and not field.name.lower().endswith("_id"):
                # ZT: Skip sensitive columns — do not create measures on PII/PHI data
                if is_sensitive_column(field.name):
                    log_audit_event("MEASURE_EXCLUDED", f"{table_name}.{field.name}",
                        "PII/PHI column excluded from auto-measures", "INFO")
                    continue

                col_label = field.name.replace("_", " ").title()
                measures.append({
                    "name": f"Total {col_label}",
                    "expression": f"SUM('{table_name}'[{field.name}])",
                })
                measures.append({
                    "name": f"Avg {col_label}",
                    "expression": f"AVERAGE('{table_name}'[{field.name}])",
                })
    return measures

# Build model tables
model_tables = []
all_measures_count = 0
excluded_sensitive_count = 0

for table_name, schema in table_schemas.items():
    current_table = table_name  # for build_tmsl_column context
    columns = [build_tmsl_column(f) for f in schema.fields]
    measures = generate_measures(table_name, schema)
    all_measures_count += len(measures)

    table_def = {
        "name": table_name,
        "columns": columns,
        "partitions": [{
            "name": f"{table_name}_partition",
            "source": {
                "type": "entity",
                "entityName": table_name,
                "schemaName": "dbo",
                "expressionSource": f"DatabaseQuery",
            }
        }]
    }
    if measures:
        table_def["measures"] = measures
    model_tables.append(table_def)

# Auto-detect relationships
relationships = detect_relationships(table_schemas, DIM_TABLES)
tmsl_relationships = [
    {
        "name": r["name"],
        "fromTable": r["fromTable"],
        "fromColumn": r["fromColumn"],
        "toTable": r["toTable"],
        "toColumn": r["toColumn"],
        "crossFilteringBehavior": "singleDirection",
    }
    for r in relationships
]

# Assemble TMSL model
tmsl_model = {
    "name": SEMANTIC_MODEL_NAME,
    "compatibilityLevel": 1604,
    "model": {
        "defaultMode": "import",
        "tables": model_tables,
        "relationships": tmsl_relationships,
    }
}

log_audit_event("SEMANTIC_MODEL_GENERATED", SEMANTIC_MODEL_NAME,
    f"{len(model_tables)} tables, {len(tmsl_relationships)} rels, {all_measures_count} measures")

print(f"✅ TMSL model definition generated (ZT: PII columns excluded from measures).")
print(f"   Tables:         {len(model_tables)}")
print(f"   Relationships:  {len(tmsl_relationships)}")
print(f"   Measures:       {all_measures_count}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PREVIEW: AUTO-DETECTED RELATIONSHIPS
# ════════════════════════════════════════════════════════════════════════

if relationships:
    print(f"Auto-detected {len(relationships)} relationships:\n")
    rel_df = pd.DataFrame(relationships)
    print(rel_df[["fromTable", "fromColumn", "toTable", "toColumn"]].to_string(index=False))
else:
    print("No auto-detected relationships (check FK naming conventions: columns ending with _id).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE SEMANTIC MODEL IN FABRIC
# ════════════════════════════════════════════════════════════════════════

import sempy.fabric as fabric
import base64

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')

FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"
url = f"{FABRIC_API_BASE}/workspaces/{workspace_id}/items"
headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json",
}

# Encode TMSL as base64 for the definition payload
tmsl_json = json.dumps(tmsl_model)
tmsl_b64 = base64.b64encode(tmsl_json.encode("utf-8")).decode("utf-8")

payload = {
    "displayName": SEMANTIC_MODEL_NAME,
    "description": f"Auto-generated semantic model for {INDUSTRY} industry",
    "type": "SemanticModel",
    "definition": {
        "parts": [
            {
                "path": "model.bim",
                "payload": tmsl_b64,
                "payloadType": "InlineBase64"
            }
        ]
    }
}

print(f"Creating semantic model: {SEMANTIC_MODEL_NAME}...")
import requests
resp = requests.post(url, headers=headers, json=payload)

if resp.status_code in (200, 201, 202):
    print(f"\n✅ Semantic model created: {SEMANTIC_MODEL_NAME}")
    resp_json = resp.json()
    if "id" in resp_json:
        print(f"   Item ID: {resp_json['id']}")
else:
    print(f"\n⚠ Creation returned status {resp.status_code}")
    print(f"   Response: {resp.text}")
    print(f"\n   Alternative: Import the TMSL definition manually via Power BI Desktop.")
    print(f"   The TMSL JSON has been generated and can be saved for manual import.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SEMANTIC MODEL SUMMARY
# ════════════════════════════════════════════════════════════════════════

print(f"\n{'═'*65}")
print(f"SEMANTIC MODEL SUMMARY — {INDUSTRY.upper()}")
print(f"{'═'*65}")
print(f"\n  Model Name:      {SEMANTIC_MODEL_NAME}")
print(f"  Data Source:     {LAKEHOUSE_NAME if SEMANTIC_MODEL_SOURCE == 'lakehouse' else WAREHOUSE_NAME} ({SEMANTIC_MODEL_SOURCE})")
print(f"")
print(f"  Star Schema:")
print(f"    Dimensions:    {len([t for t in table_schemas if t.startswith('dim_')])}")
print(f"    Facts:         {len([t for t in table_schemas if not t.startswith('dim_')])}")
print(f"    Relationships: {len(tmsl_relationships)}")
print(f"    Measures:      {all_measures_count}")
print(f"")
print(f"  Tables in model:")
for t in model_tables:
    cat = "DIM" if t['name'].startswith('dim_') else "FACT"
    m_count = len(t.get('measures', []))
    m_text = f"  ({m_count} measures)" if m_count > 0 else ""
    print(f"    [{cat}] {t['name']:<40} {len(t['columns'])} cols{m_text}")

print(f"\n✅ Semantic model creation complete.")
print(f"   Next: Run 05_Create_Ontology.ipynb to create the industry ontology.")